## Chunking OCR Text

The goal of this script is to split long OCR-extracted texts into smaller, manageable chunks of 500 tokens with 50-token overlap. This makes the text suitable for embedding and retrieval in a Retrieval-Augmented Generation (RAG) pipeline.


### Part 1 – Chunking Text from Tesseract OCR

**Step 1: Import Required Libraries**

In [1]:
# Import necessary libraries
import os                      # For file path management
import pandas as pd            # For handling CSVs and tables
from nltk.tokenize import word_tokenize  # To break text into words (tokens)
from tqdm import tqdm          # To show a progress bar while processing

**Step 2: Load Cleaned Tesseract Text**

In [2]:
# Path to your cleaned & merged Tesseract text file
csv_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\extracted_text_Tesseract\cleaned_text_Tesseract\merged_cleaned_text_Tesseract.csv"

# Read the CSV file into a DataFrame (like a table in memory)
df = pd.read_csv(csv_path)

**Step 3: Define Chunking Parameters**

In [3]:
# Each chunk will have 500 tokens.
chunk_size = 500      
# Number of overlapping tokens between chunks. Each chunk overlaps 50 tokens with the previous one to preserve context.
overlap = 50          

# Prepare an empty list to store each chunk
chunks = []
# Start counter for chunk IDs
chunk_id = 0          

**Step 4: Chunk Each Text Document**

In [4]:
# Iterate over each row in the DataFrame
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # Name of the original text file
    filename = row['filename']           
    # The actual text content (ensure it's a string)
    text = str(row['text'])              
    # Split the text into words (tokens)
    tokens = word_tokenize(text)         

    # Create overlapping chunks
    for i in range(0, len(tokens), chunk_size - overlap):
        # Select 500 tokens with overlap
        chunk_tokens = tokens[i:i + chunk_size]          
        # Combine tokens into a string
        chunk_text = ' '.join(chunk_tokens)              

        # Make sure chunk isn't empty
        if chunk_text.strip():                           
            chunks.append({
                # Create a unique ID for this chunk
                'chunk_id': f"{filename}_{chunk_id}",    
                # Track original file name
                'filename': filename,                    
                # Store the actual chunk text
                'chunk_text': chunk_text                 
            })
            # Increment chunk ID
            chunk_id += 1                                



100%|██████████| 9053/9053 [00:26<00:00, 340.41it/s]


**Step 5: Save Chunks to CSV**

In [5]:
#Where to save the chunked text
output_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\extracted_text_Tesseract\cleaned_text_Tesseract\chunked_text_Tesseract.csv"

# Convert list of chunks into a DataFrame
chunk_df = pd.DataFrame(chunks)

# Save to CSV
chunk_df.to_csv(output_path, index=False)

chunk_df.head(n=4)

,chunk_id,filename,chunk_text
0,0_0.txt_0,0_0.txt,% wikipedia vl donate create account log in 9 ...
1,0_1.txt_1,0_1.txt,animation { ecit ] year title role notes
2,0_10.txt_2,0_10.txt,"this page was last edited on 20 april 2025 , a..."
3,0_100.txt_3,0_100.txt,billy joel 's parents met in the late 1930s at...


### Part 2 – Chunking Text from PaddleOCR

The structure is identical to the Tesseract chunking, with only the CSV input/output path changed.



In [6]:
# Used for handling file paths (not strictly needed here but a good habit)
import os  
# Used to work with CSV files and data tables
import pandas as pd  
# Used to split text into individual words (tokens)
from nltk.tokenize import word_tokenize  
# Used to show a progress bar during long operations
from tqdm import tqdm  


**Step 2: Load Cleaned Tesseract Text**

In [7]:
# Path to the cleaned and merged PaddleOCR CSV file
csv_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\extracted_text_PaddleOCR2\cleaned_text_PaddleOCR2\merged_cleaned_text_PaddleOCR2.csv"
# Load the CSV into a DataFrame
df = pd.read_csv(csv_path)

**Step 3: Define Chunking Parameters**

In [8]:
# Each chunk will contain 500 tokens (words)
chunk_size = 500      
# 50 tokens from the end of one chunk will appear at the start of the next (to preserve context)
overlap = 50          

# Prepare output structure
# This will store the final list of chunks
chunks = []          
# This is a counter to give each chunk a unique ID
chunk_id = 0         



**Step 4: Chunk Each Text Document**

In [9]:
# Go through each row (i.e., each document) in the CSV
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # Get the name of the file the text came from
    filename = row['filename']       
    # Get the text from that row (and make sure it's a string)
    text = str(row['text'])          
    # Split the text into individual words (tokens)
    tokens = word_tokenize(text)     

    # Split the tokens into chunks with overlap
    for i in range(0, len(tokens), chunk_size - overlap):
        # Take 500 tokens, starting every (500 - 50 = 450)
        chunk_tokens = tokens[i:i + chunk_size]         
        # Join tokens back into a string
        chunk_text = ' '.join(chunk_tokens)             
        # Only keep the chunk if it contains non-empty text
        if chunk_text.strip():                          
            chunks.append({
                # Create a unique ID for the chunk
                'chunk_id': f"{filename}_{chunk_id}",   
                # Store the original filename
                'filename': filename,                   
                # Store the actual chunk of text
                'chunk_text': chunk_text                
            })
            chunk_id += 1

100%|██████████| 9100/9100 [00:24<00:00, 371.66it/s]


**Step 5: Save Chunks to CSV**

In [10]:
# Define the path where the final chunked CSV will be saved
output_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\extracted_text_PaddleOCR2\cleaned_text_PaddleOCR2\chunked_text_PaddleOCR.csv"
# Convert the chunks list into a DataFrame (table)
chunk_df = pd.DataFrame(chunks)
# Save the table into a CSV file without the index column
chunk_df.to_csv(output_path, index=False)

chunk_df.head(n=3)

,chunk_id,filename,chunk_text
0,0_0.txt_0,0_0.txt,[ WIKIPEDIA Donate Create account Log in . The...
1,0_1.txt_1,0_1.txt,[ Animation [ edit ] Year Title Role Notes 199...
2,0_10.txt_2,0_10.txt,"[ This page was last edited on 20 April 2025 ,..."


Both versions create a CSV file with the following columns:

+ chunk_id: Unique ID (e.g., filename_0, filename_1, ...)
+ filename: Name of the source document
+ chunk_text: The actual chunk of 500 tokens (with overlap)